In [ ]:
#importing libraries
import os
import re
import sklearn
import pandas as pd
from tqdm.notebook import tqdm, trange
from matplotlib import pyplot as plt
import numpy
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score, auc, roc_curve, RocCurveDisplay
from sklearn.metrics import confusion_matrix,classification_report
import seaborn as sns
import tensorflow_hub as hub
from tensorflow.keras.layers import Input,Flatten, Lambda,LSTM, Bidirectional, Dense, Dropout, BatchNormalization, Conv2D,MaxPooling1D,Conv1D, GRU
from tensorflow.keras.models import Model
from tensorflow import keras
from tensorflow.keras import regularizers
import tensorflow as tf
from keras.layers import concatenate
from tensorflow.keras.utils import plot_model
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint, EarlyStopping 
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import StratifiedKFold
from sklearn.utils import class_weight


In [ ]:
#loading datasets
trainData=pd.read_csv("rabies_lyssavirus_training_dataset_preproccessed.csv")
testData=pd.read_csv("rabies_lyssavirus_test_dataset_preproccessed.csv")


In [ ]:
#encoding values.
trainDataset_3D = []
testDataset_3D = []

for i in trange(len(trainData)):
    sequence=[]
    for x in trainData['SEQUENCE'][i]:
        if x=='A':
            sequence.append([1,0,0,0,0])
        elif x=='C':
            sequence.append([0,1,0,0,0])
        elif x=='G':
            sequence.append([0,0,1,0,0])
        elif x=='T':
            sequence.append([0,0,0,1,0])
        elif x=='N':
            sequence.append([0,0,0,0,1])
    trainDataset_3D.append(sequence)
for i in trange(len(testData)):
    sequence=[]
    for x in testData['SEQUENCE'][i]:
        if x=='A':
            sequence.append([1,0,0,0,0])
        elif x=='C':
            sequence.append([0,1,0,0,0])
        elif x=='G':
            sequence.append([0,0,1,0,0])
        elif x=='T':
            sequence.append([0,0,0,1,0])
        elif x=='N':
            sequence.append([0,0,0,0,1])
    testDataset_3D.append(sequence)

trainDataset_3D = numpy.array(trainDataset_3D)
testDataset_3D = numpy.array(testDataset_3D)

In [ ]:
host_map = {
    
    'Canis lupus familiaris'   :0,       
    'Bos taurus'               :1,                  
    'Vulpes vulpes'            :2,             
    'Procyon lotor'            :3,               
    'Desmodus rotundus'        :4,          
    'Mephitis mephitis'        :5,           
    'Homo sapiens'             :6,             
    'Eptesicus fuscus'         :7,           
    'Canidae'                  :8,             
    'Mephitidae'               :9,                
    'Felis catus'              :10,                 
    'Vulpes lagopus'           :11,             
    'Tadarida brasiliensis'    :12,
    'Melogale'                 :13,              
    'Equus caballus'           :14,             
    'Canis mesomelas'          :15,             
    'Otocyon megalotis'        :16,            
    'Artibeus lituratus'       :17,        
    'Capra hircus'             :18,           
    'Canis'                    :19,                    
    'Nyctereutes procyonoides' :20,     
    'Chiroptera'               :21,            
    'Aeorestes cinereus'       :22,         
    'Ovis aries'               :23,                 
    'Lasiurus borealis'        :24,            
    'Axis axis'                :25,              
    'Cerdocyon thous'          :26,              
    'Herpestidae'              :27,              
    'Lasionycteris noctivagans':28, 

    0:'Canis lupus familiaris',       
    1:'Bos taurus',                  
    2:'Vulpes vulpes',             
    3:'Procyon lotor',               
    4:'Desmodus rotundus',          
    5:'Mephitis mephitis',           
    6:'Homo sapiens',             
    7:'Eptesicus fuscus',           
    8:'Canidae',             
    9:'Mephitidae',                
    10:'Felis catus',                 
    11:'Vulpes lagopus',             
    12:'Tadarida brasiliensis',
    13:'Melogale',              
    14:'Equus caballus',             
    15:'Canis mesomelas',             
    16:'Otocyon megalotis',            
    17:'Artibeus lituratus',        
    18:'Capra hircus',           
    19:'Canis',                    
    20:'Nyctereutes procyonoides',     
    21:'Chiroptera',            
    22:'Aeorestes cinereus',         
    23:'Ovis aries',                 
    24:'Lasiurus borealis',            
    25:'Axis axis',              
    26:'Cerdocyon thous',              
    27:'Herpestidae',              
    28:'Lasionycteris noctivagans',
    
}
maps = {
    'HOST' : host_map,
}

In [ ]:
#converting host labels to numerical value
trainData['HOST'] = trainData['HOST'].apply(lambda x : maps["HOST"][x])
testData['HOST'] = testData['HOST'].apply(lambda x : maps["HOST"][x])


In [ ]:
kfold = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

In [ ]:
def build_model():
            input_layer = Input(shape=trainDataset_3D.shape[1:], name="input_layer")
            BiLSTM_Model = Bidirectional(keras.layers.LSTM(128, return_sequences= False), name="BiLSTM_Model")(input_layer)
            DROPOUT = Dropout(0.2, name="DROPOUT")(BiLSTM_Model)
            BatchNormalized = BatchNormalization(axis = -1, name="BatchNormalized")(DROPOUT)
            Dense_Layer1 = Dense(64, activation='relu', name="Dense_Layer1")(BatchNormalized)
            output = Dense(29, activation='softmax', name="output")(Dense_Layer1)                                                                                                                                                       
            model = Model(inputs=[input_layer], outputs=output, name="BILSTM")
            model.summary()
            model.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['accuracy'])
            return model

In [ ]:
fold_no = 1

#creating and training model with 5 fold
for train, val in kfold.split(trainData['SEQUENCE'], trainData['HOST']):

    trainingDataKfold = trainDataset_3D[numpy.ix_(train)]
    valDataKfold = trainDataset_3D[numpy.ix_(val)]
    tragetTrainingData = []
    targetValData = []
    
    for i in train:
        tragetTrainingData.append(trainData['HOST'][i])
    tragetTrainingData = numpy.array(tragetTrainingData)

    for i in val:
        targetValData.append(trainData['HOST'][i])
    targetValData = numpy.array(targetValData)
    
    
    rabies_model = build_model()
    checkpoint = ModelCheckpoint('rabies_lyssavirus_best_weight_Fold_'+str(fold_no)+'_Epoch-{epoch:03d}-valACC-{val_accuracy:.4f}.h5', 
        verbose=1, 
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=True,
        mode='max'
    )

    model_rabies = rabies_model.fit([trainingDataKfold], tragetTrainingData,validation_data=([valDataKfold], targetValData), epochs=200, batch_size=128,callbacks=[checkpoint])
    rabies_model.save_weights('BILSTM_Class_Balanced_rabies_lyssavirus_Fold_'+str(fold_no)+'_Epoch1to200.h5')
    fold_no = fold_no+1

In [ ]:
#testing the model
rabies_model = build_model()
checkpoint = ModelCheckpoint('rabies_lyssavirus_best_weight_Fold_'+str(fold_no)+'_Epoch-{epoch:03d}-valACC-{val_accuracy:.4f}.h5', 
        verbose=1, 
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=True,
        mode='max'
)

rabies_model.load_weights('rabies_lyssavirus_best_weight_Fold_1_Epoch-124-valACC-0.7846.h5')
testCore = rabies_model.evaluate(testDataset_3D, testData['HOST'], batch_size=128)
print(testCore)

In [ ]:
category_list = [ 'Canis lupus familiaris',       
    'Bos taurus',                  
    'Vulpes vulpes',             
    'Procyon lotor',               
    'Desmodus rotundus',          
    'Mephitis mephitis',           
    'Homo sapiens',             
    'Eptesicus fuscus',           
    'Canidae',             
    'Mephitidae',                
    'Felis catus',                 
    'Vulpes lagopus',             
    'Tadarida brasiliensis',
    'Melogale',              
    'Equus caballus',             
    'Canis mesomelas',             
    'Otocyon megalotis',            
    'Artibeus lituratus',        
    'Capra hircus',           
    'Canis',                    
    'Nyctereutes procyonoides',     
    'Chiroptera',            
    'Aeorestes cinereus',         
    'Ovis aries',                 
    'Lasiurus borealis',            
    'Axis axis',              
    'Cerdocyon thous',              
    'Herpestidae',              
    'Lasionycteris noctivagans', 
]

In [ ]:
predictions = rabies_model.predict(testDataset_3D, batch_size=128, verbose=1)

In [ ]:
def get_prediction_labels(prediction_probabilities):
    return numpy.argmax(prediction_probabilities)

def get_labels():
    actual_label=[]
    for i in trange(testData['HOST'].shape[0]):
        actual_label.append(testData['HOST'][i])
    return actual_label


In [ ]:
predicted_labels = [get_prediction_labels(prediction) for prediction in tqdm(predictions)]
actual_labels = get_labels()

In [ ]:
print(classification_report(actual_labels, predicted_labels))

In [ ]:
#generating confusion matrix
cm = confusion_matrix(actual_labels, predicted_labels)
cmn = cm.astype('float')/ cm.sum(axis=1)[:, numpy.newaxis]
fig, ax = plt.subplots(figsize=(36,20))
sns.heatmap(cmn, annot=True, fmt='.4f',cmap="Oranges", xticklabels=category_list, yticklabels=category_list)
plt.ylabel('True Host', fontsize=12)
plt.xlabel('Predicted Host', fontsize=12)
plt.savefig('rabies_confusion_matrix.png', dpi=600, 
         format='png',bbox_inches='tight'
        )
plt.show(block=False)

In [ ]:
label_binarizer = LabelBinarizer().fit(trainData['HOST'])
y_onehot_test = label_binarizer.transform(testData['HOST'])
y_onehot_test.shape

In [ ]:
#multiclass roc curve
fpr = {} # False Positive Rate
tpr = {} # True Positive Rate
thresh ={} # Threshold
roc_auc = dict()

plt.figure(figsize = (15, 9))
for i in range(29):
    fpr[i], tpr[i], thresh[i] = roc_curve(y_onehot_test[:, i], predictions[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
    
    plt.plot(fpr[i], tpr[i], linestyle='--', 
             label='%s vs Rest (AUC=%0.2f)'%(category_list[i],roc_auc[i]))


plt.plot([0,1],[0,1],'b--')
plt.xlim([0,1])
plt.ylim([0,1.05])
plt.title('Multiclass ROC curve of Rabies Lyssavirus')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.legend(loc='lower right')
plt.savefig('rabies_roc_curve.png', dpi=1200, 
         format='png',bbox_inches='tight'
        )
plt.show()

In [ ]:
micro_roc_auc_ovr = roc_auc_score(
    y_onehot_test,
    predictions,
    multi_class="ovr",
    average="micro",
)

print(f"Micro-averaged One-vs-Rest ROC AUC score:\n{micro_roc_auc_ovr:.4f}")